### 📦 Import Core Libraries

These libraries cover numerical operations, plotting, system utilities, and data handling.

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
from contextlib import contextmanager
import re
import pandas
import pathlib

### ⚙️ Load GROMACS and Dependencies

> **Note:**  
> These scripts are designed for **ACCRE**.  
> If you are using a different HPC, adjust the module load commands to match your system setup.

#### 🧩 Option 1 – Adjust for Your HPC
Edit the `module load` lines below to match the versions of **GROMACS**, **OpenMPI**, **CUDA**, or other dependencies available on your cluster.

#### 🧪 Option 2 – Use a Manual GROMACS Path
If your cluster does **not** use module environments, remove the `module load` commands and instead source your GROMACS installation manually:
```bash
source /path/to/gromacs/bin/GMXRC


In [11]:
import os
import subprocess

def load_gmx_plumed():
    cmd = """
    module load StdEnvACCRE/2023
    module load gcc/12.3
    module load openmpi/4.1.5
    module load cuda/12.2
    source $HOME/gromacs-2023.5-plumed-mpi/bin/GMXRC
    export PLUMED_KERNEL=$HOME/plumed-2.9.2/lib/libplumedKernel.so
    export LD_LIBRARY_PATH=$HOME/plumed-2.9.2/lib:$LD_LIBRARY_PATH
    export PATH=$HOME/plumed-2.9.2/bin:$PATH
    env
    """
    # run in bash, capture environment
    result = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    for line in result.stdout.splitlines():
        key, _, value = line.partition("=")
        os.environ[key] = value


In [12]:
load_gmx_plumed()

### 🧬 Install SMOG2 (if not already installed)

> **Note:**  
> ACCRE uses **Apptainer** for containerized applications.  
> If your system uses a different container tool, adjust accordingly and install **SMOG2** as per your environment.


In [4]:
%%bash
cd ~
apptainer pull smog2.sif docker://smogserver/smog2:latest

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

CalledProcessError: Command 'b'cd ~\napptainer pull smog2.sif docker://smogserver/smog2:latest\n'' returned non-zero exit status 255.

### Verify SMOG2 Installation

Run the following to confirm that **SMOG2** is accessible inside the Apptainer container.  
This prints the first few lines of the `--help` output.

In [13]:
%%bash
apptainer exec ~/smog2.sif smog2 --help | head -n 10

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

*******************************************************************************************

******* ******* ******* ******* ******* SMOG v2.4.5 ******* ******* ******* ******* *******
                                             
               Thank you for using the Structure-based Model (SMOG) software
                                             
     This package is the product of contributions from a number of people, including:
            Jeffrey Noel, Mariana Levi, Antonio Oliveira, Vinícius Contessoto,
          Esteban Dodero-Rojas, Mohit Raghunathan, Joyce Yang, Prasad Bandarkar,
                  Udayan Mohanty, Ailun Wang, Heiko Lammert, Ryan Hayes,


Create three directories — **3PRG**, **6ONI**, and **6ONJ** — and place the corresponding PDB files inside them, named **3PRG.pdb**, **6ONI.pdb**, and **6ONJ.pdb** respectively.

In [ ]:
!mkdir 3PRG
!mkdir 6ONJ
!mkdir 6ONI

Navigate to the directory that contains the required files and where you want the output files to be saved

In [ ]:
%cd 3PRG

### ⚗️ Build SMOG2 All-Atom Model (ACCRE Workflow)

This workflow prepares and builds a **SMOG2 all-atom structure-based model** inside the **Apptainer** container on ACCRE.

---

#### 🧩 Steps Overview
1. **Clean the PDB** – Keep only `ATOM`, `TER`, and `END` records to remove ligands, waters, and other non-protein entries.  
2. **Adjust PDB Format** – Use `smog_adjustPDB` inside the container to fix residue numbering and ensure SMOG2 compatibility.  
3. **Generate the Model** – Run `smog2` with the `-AA` flag to build an all-atom GROMACS topology (`.gro`, `.top`, etc.).  
4. **Organize Outputs** – Move generated files into a dedicated `AA` folder for clarity.  
5. **Verify** – Check the generated files to ensure successful SMOG2 processing.

---

> **Note:**  
> ACCRE uses **Apptainer**; if your system uses another container engine, adjust the commands accordingly.  
> Ensure the SMOG2 `.sif` file and your working directory are properly bound when executing.


In [9]:
%%bash
# === Step 1: Clean the PDB file ===
# Keep only the relevant structural records:
#  - ATOM: standard protein atoms
#  - TER: terminators between chains
#  - END: file termination
# Removes any HETATM (waters, ligands) and metadata lines that SMOG2 does not need.
grep -E "^(ATOM|TER|END)" 3PRG.pdb  > 3PRG-clean.pdb

# === Step 3: Adjust the PDB format using SMOG2 ===
# smog_adjustPDB fixes residue numbering, adds missing TER if needed,
# and ensures the file follows SMOG2 format conventions.
# -i : input file
# -o : output adjusted file
# We run this inside the Apptainer container so we don't need to install SMOG2 manually.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog_adjustPDB -i /work/3PRG-clean.pdb -o /work/3PRG-adjusted.pdb > out-smog_adjustPBD.dat

# === Step 4: Generate the All-Atom Structure-Based Model ===
# smog2 creates the GROMACS forcefield/topology from the adjusted PDB.
# -i : input PDB
# -dname : base name for all generated files (6ONI-AA.gro, 6ONI-AA.top, etc.)
# -AA : tells SMOG2 to build an all-atom model (instead of CA coarse-grained model)
# Output is logged into out-smog2-AA.dat for later inspection.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog2 -i /work/3PRG-adjusted.pdb -dname 3PRG-AA -AA &>> out-smog2-AA.dat



bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

In [16]:
%%bash
gmx_mpi editconf -f 3PRG-AA.gro -c -d 10 -bt cubic -o 3PRG-AA-box.gro &>> out-editconf-AA.log

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

In [ ]:
%cd 6ONJ

In [ ]:
%%bash
# === Step 1: Clean the PDB file ===
# Keep only the relevant structural records:
#  - ATOM: standard protein atoms
#  - TER: terminators between chains
#  - END: file termination
# Removes any HETATM (waters, ligands) and metadata lines that SMOG2 does not need.
grep -E "^(ATOM|TER|END)" 6ONJ.pdb  > 6ONJ-clean.pdb

# === Step 3: Adjust the PDB format using SMOG2 ===
# smog_adjustPDB fixes residue numbering, adds missing TER if needed,
# and ensures the file follows SMOG2 format conventions.
# -i : input file
# -o : output adjusted file
# We run this inside the Apptainer container so we don't need to install SMOG2 manually.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog_adjustPDB -i /work/6ONJ-clean.pdb -o /work/6ONJ-adjusted.pdb > out-smog_adjustPBD.dat

# === Step 4: Generate the All-Atom Structure-Based Model ===
# smog2 creates the GROMACS forcefield/topology from the adjusted PDB.
# -i : input PDB
# -dname : base name for all generated files (6ONJ-AA.gro, 6ONJ-AA.top, etc.)
# -AA : tells SMOG2 to build an all-atom model (instead of CA coarse-grained model)
# Output is logged into out-smog2-AA.dat for later inspection.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog2 -i /work/6ONJ-adjusted.pdb -dname 6ONJ-AA -AA &>> out-smog2-AA.dat



In [ ]:
%%bash
gmx_mpi editconf -f 6ONJ-AA.gro -c -d 10 -bt cubic -o 6ONJ-AA-box.gro &>> out-editconf-AA.log

In [ ]:
%cd 6ONI

In [ ]:
%%bash
# === Step 1: Clean the PDB file ===
# Keep only the relevant structural records:
#  - ATOM: standard protein atoms
#  - TER: terminators between chains
#  - END: file termination
# Removes any HETATM (waters, ligands) and metadata lines that SMOG2 does not need.
grep -E "^(ATOM|TER|END)" 6ONI.pdb  > 6ONI-clean.pdb

# === Step 3: Adjust the PDB format using SMOG2 ===
# smog_adjustPDB fixes residue numbering, adds missing TER if needed,
# and ensures the file follows SMOG2 format conventions.
# -i : input file
# -o : output adjusted file
# We run this inside the Apptainer container so we don't need to install SMOG2 manually.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog_adjustPDB -i /work/6ONI-clean.pdb -o /work/6ONI-adjusted.pdb > out-smog_adjustPBD.dat

# === Step 4: Generate the All-Atom Structure-Based Model ===
# smog2 creates the GROMACS forcefield/topology from the adjusted PDB.
# -i : input PDB
# -dname : base name for all generated files (6ONI-AA.gro, 6ONI-AA.top, etc.)
# -AA : tells SMOG2 to build an all-atom model (instead of CA coarse-grained model)
# Output is logged into out-smog2-AA.dat for later inspection.
apptainer exec --bind $(pwd):/work ~/smog2.sif \
    smog2 -i /work/6ONI-adjusted.pdb -dname 6ONI-AA -AA &>> out-smog2-AA.dat



### 🌡️ Prepare Simulation Boxes and Temperature Windows

Sets up a **10 nm cubic box** and generates `.mdp` files for a series of **temperature runs**.

---

#### 🔧 Steps
1. cd into each system (3PRG, 6ONJ & 6ON)
2. Use `gmx_mpi editconf` to define a 10 nm cubic simulation box.  
3. Automatically generate `.mdp` files for multiple temperatures (100–140 K).  
4. You can modify the temperature list to explore your own range or investigate specific thermal effects.

---

> **Note:**  
> This uses **host GROMACS (`gmx_mpi`)**, not the container.  
> Ensure the `example.mdp` template path is correct for your setup.  
> You may need to edit the `gmx_mpi` command depending on how your GROMACS was compiled—  
> for example, on ACCRE we use the **MPI-compiled version**, hence the executable is `gmx_mpi`.


In [ ]:
%%bash

cd 3PRG

# 1️⃣ Define a cubic box with 10 nm margin (using host gmx_mpi, not container)
gmx_mpi editconf -f ../3PRG-AA.gro -c -d 10 -bt cubic -o 3PRG-AA-box.gro &>> out-editconf-AA.log

# 2️⃣ Generate reference.mdp files for multiple temperatures
temperatures="100 102 104 106 107 108 110 112 114 116 118"
steps="10000000"

for T in $temperatures; do
  mkdir -p $T
  sed "s/TEMP/${T}/g; s/TSTEPS/${steps}/g" ../../example.mdp > $T/reference.mdp
done

echo "✅ Box created and MDP files prepared for: $temperatures"

../


cd 6ONJ

# 1️⃣ Define a cubic box with 10 nm margin (using host gmx_mpi, not container)
gmx_mpi editconf -f ../6ONJ-AA.gro -c -d 10 -bt cubic -o 3PRG-AA-box.gro &>> out-editconf-AA.log

# 2️⃣ Generate reference.mdp files for multiple temperatures
temperatures="100 102 104 106 107 108 110 112 114 116 118"
steps="10000000"

for T in $temperatures; do
  mkdir -p $T
  sed "s/TEMP/${T}/g; s/TSTEPS/${steps}/g" ../../example.mdp > $T/reference.mdp
done

echo "✅ Box created and MDP files prepared for: $temperatures"
    
../

cd 6ONI

# 1️⃣ Define a cubic box with 10 nm margin (using host gmx_mpi, not container)
gmx_mpi editconf -f ../3PRG-AA.gro -c -d 10 -bt cubic -o 3PRG-AA-box.gro &>> out-editconf-AA.log

# 2️⃣ Generate reference.mdp files for multiple temperatures
temperatures="100 102 104 106 107 108 110 112 114 116 118"
steps="10000000"

for T in $temperatures; do
  mkdir -p $T
  sed "s/TEMP/${T}/g; s/TSTEPS/${steps}/g" ../../example.mdp > $T/reference.mdp
done

echo "✅ Box created and MDP files prepared for: $temperatures"


### ⚙️ Generate `.tpr` Files for Each Temperature

This phase compiles GROMACS input files (`.tpr`) for each temperature window, preparing the system for simulation or replica exchange runs.

---

#### 🔧 Steps
1. cd into every system (3PRG, 6ONJ & 6ONI)
2. Loop through each temperature directory (100–118 K).  
3. Run `gmx_mpi grompp` to create a `.tpr` file (`t-*.tpr`) using:
   - `reference.mdp` – simulation parameters  
   - `*-AA-box.gro` – boxed structure  
   - `*-AA.top` – topology file  
4. Logs are stored in each temperature folder as `grompp.log`.

---

> **Note:**  
> Ensure that `gmx_mpi` matches your GROMACS build.  
> On **ACCRE**, we use the **MPI-compiled version** (`gmx_mpi`), but some systems may use `gmx` instead.


In [ ]:
%%bash
# === Phase 3: Build .tpr files for each temperature ===
cd 3PRG

for T in 100 102 104 106 108 110 112 114 116 118 ; do
  cd $T
  # One-line grompp with log inside each folder
  gmx_mpi grompp -f reference.mdp -c ../3PRG-AA-box.gro -p ../../3PRG-AA.top -o t-3PRG.tpr &>> grompp.log
  cd ..
done

../ 

cd 6ONJ

for T in 100 102 104 106 108 110 112 114 116 118 ; do
  cd $T
  # One-line grompp with log inside each folder
  gmx_mpi grompp -f reference.mdp -c ../6ONJ-AA-box.gro -p ../../6ONJ-AA.top -o t-6ONJ.tpr &>> grompp.log
  cd ..
done

../

cd 6ONI 

for T in 100 102 104 106 108 110 112 114 116 118 ; do
  cd $T
  # One-line grompp with log inside each folder
  gmx_mpi grompp -f reference.mdp -c ../6ONI-AA-box.gro -p ../../6ONI-AA.top -o t-6ONI.tpr &>> grompp.log
  cd ..
done


echo "✅ .tpr files and logs created in each folder"


# SLURM SCRIPT TO RUN ALL SIMULATIONS 
This SLURM script submits sequential HREMD simulations for all three systems — **6ONJ**, **6ONI**, and **3PRG**.  
Each system is run one after another using the same MPI/GPU settings, ensuring identical temperature ladders and consistent sampling across the full SMOG replica-exchange workflow.  
Once all three runs finish, the job reports successful completion.


In [ ]:
%%bash
cat > SMOG_all_temp_run.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=SMOG_All
#SBATCH --mail-user=
#SBATCH --mail-type=ALL
#SBATCH --nodes=1
#SBATCH --ntasks=10
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --time=05-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:2
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=SMOG_all_out_%j.log
#SBATCH --error=SMOG_all_err_%j.log

# === Load environment ===
module load StdEnvACCRE/2023
module load gcc/12.3 openmpi/4.1.5 cuda/12.2
source $HOME/gromacs-2023.5-plumed-mpi/bin/GMXRC

# === PLUMED environment ===
export PLUMED_KERNEL=$HOME/plumed-2.9.2/lib/libplumedKernel.so
export LD_LIBRARY_PATH=$HOME/plumed-2.9.2/lib:$LD_LIBRARY_PATH
export PATH=$HOME/plumed-2.9.2/bin:$PATH

export OMP_NUM_THREADS=1
export GMX_DISABLE_GPU_TIMING=1
export GMX_ENABLE_DIRECT_GPU_COMM=1
unset CUDA_VISIBLE_DEVICES

# === Sequential runs for 6ONJ → 6ONI → 3PRG ===
cd 6ONJ
echo "🚀 Launching NEW GROMACS HREMD run for 6ONJ ..."
mpirun --oversubscribe -np 10 gmx_mpi mdrun \
  -multidir 100 102 104 106 108 110 112 114 116 118 \
  -s t-6ONJ.tpr -pin auto -ntomp 1 -v \
  -nsteps 100000000 -replex 200 \
  -nb gpu -bonded gpu \
  -gputasks "0000011111"
echo "✅ Completed 6ONJ"
cd ..

cd 6ONI
echo "🚀 Launching NEW GROMACS HREMD run for 6ONI ..."
mpirun --oversubscribe -np 10 gmx_mpi mdrun \
  -multidir 100 102 104 106 108 110 112 114 116 118 \
  -s t-6ONI.tpr -pin auto -ntomp 1 -v \
  -nsteps 100000000 -replex 200 \
  -nb gpu -bonded gpu \
  -gputasks "0000011111"
echo "✅ Completed 6ONI"
cd ..

cd 3PRG
echo "🚀 Launching NEW GROMACS HREMD run for 3PRG ..."
mpirun --oversubscribe -np 10 gmx_mpi mdrun \
  -multidir 100 102 104 106 108 110 112 114 116 118 \
  -s t-3PRG.tpr -pin auto -ntomp 1 -v \
  -nsteps 100000000 -replex 200 \
  -nb gpu -bonded gpu \
  -gputasks "0000011111"
echo "✅ Completed 3PRG"
cd ..

echo "🎉 All three systems finished successfully!"
EOF

sbatch SMOG_all_temp_run.slurm
